In [1]:
# Test-1

import yaml
from dotenv import load_dotenv
import os

import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

# Load config.yaml
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("✅ Environment loaded successfully")
print("✅ LLM Provider:", config["llm"]["provider"])
print("✅ OpenAI Key exists:", "OPENAI_API_KEY" in os.environ)
print("✅ Google Key exists:", "GOOGLE_API_KEY" in os.environ)

✅ Environment loaded successfully
✅ LLM Provider: openai
✅ OpenAI Key exists: True
✅ Google Key exists: True


In [2]:
# Test-2: Loader

from utils.loader import load_and_chunk_docs

chunks = load_and_chunk_docs("./data/raw/insurance_docs", chunk_size=80, chunk_overlap=20)
print(f"First chunk sample:\n{chunks[0].page_content[:150]}...")
print(f"Total chunks created: {len(chunks)}")

✅ Loaded 2 docs → 65 chunks
First chunk sample:
Claims Reporting and Processing Procedure

I. Initiate Your Claim...
Total chunks created: 65


In [ ]:
# Test-3: Retriever
 
from utils.loader import load_and_chunk_docs
from utils.retriever import create_retriever, load_retriever, load_config

config = load_config("config.yaml")

chunks = load_and_chunk_docs("./data/raw/insurance_docs")

# create or load provider-specific FAISS
retriever = create_retriever(chunks, provider=config["llm"]["provider"], config=config)

# load again to verify
retriever = load_retriever(provider=config["llm"]["provider"], config=config)

✅ Loaded 2 docs → 8 chunks
✅ Created new FAISS index (openai) at: ./data/embeddings/faiss_openai
✅ Loaded FAISS index (openai) from: ./data/embeddings/faiss_openai


In [ ]:
# Test-4: Query Re-writer

from utils.query_rewriter import rewrite_query
import yaml

# Confirm config is read correctly
with open("config.yaml") as f:
    config = yaml.safe_load(f)["llm"]
    print("✅ Loaded config:", config)

query = "cashless policy?"

rewritten = rewrite_query(query)
print("\n🔁 Rewritten Query:\n", rewritten)

✅ Loaded config: {'provider': 'openai', 'model_openai': 'gpt-4o-mini', 'model_gemini': 'gemini-2.5-flash', 'temperature': 0.3, 'max_tokens': 1000}

🔁 Rewritten Query:
 What are the details and implications of a cashless policy, and how does it affect consumers and businesses in today's economy?


In [6]:
# Test-5: hyde-generator

from utils.hyde_generator import generate_hyde_embedding

query = "cashless policy?"
embedding = generate_hyde_embedding(query)

print("\n✅ Embedding shape:", embedding.shape)
print("✅ Example values:", embedding[:5])


🧪 Synthetic HyDE Answer:
 A cashless policy refers to a governmental or organizational initiative aimed at reducing or eliminating the use of physical cash in transactions, promoting digital payment methods instead. This policy often encourages the adoption of technologies such as mobile wallets, credit and debit cards, and online banking to facilitate seamless transactions. Proponents argue that cashless systems enhance efficiency, reduce costs associated with handling cash, and improve tax compliance by creating a more transparent economy. Countries like Sweden and India have implemented various cashless initiatives, seeing a significant shift towards digital payments in recent years.

✅ Embedding shape: (1536,)
✅ Example values: [ 0.0236969  -0.02989197  0.02423096  0.03210449  0.0114212 ]
